ADVANCED DATA ANALYSIS IN PANDAS

In [2]:
import pandas as pd
import numpy as np

In [5]:
# Load the dataset
df = pd.read_csv("Data_Co_Supply_Chain_Dataset.csv" ,
encoding="latin1")
df.head()

,Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Name,Customer City,Customer Country,Customer Fname,Customer Id,Customer Lname,...,Order Item Quantity,Sales,Order Item Total,Order Profit Per Order,Order Region,Order State,Order Status,Product Name,shipping date (DateOrders),Shipping Mode
0,91.250000,314.640015,Advance shipping,0,Sporting Goods,Caguas,Puerto Rico,Cally,20755,Holloway,...,1,327.75,314.640015,91.250000,Southeast Asia,Java Occidental,COMPLETE,Smart watch,02-03-2018 22:56,Standard Class
1,-249.089996,311.359985,Late delivery,1,Sporting Goods,Caguas,Puerto Rico,Irene,19492,Luna,...,1,327.75,311.359985,-249.089996,South Asia,Rajastán,PENDING,Smart watch,1/18/2018 12:27,Standard Class
2,-247.779999,309.720001,Shipping on time,0,Sporting Goods,San Jose,EE. UU.,Gillian,19491,Maldonado,...,1,327.75,309.720001,-247.779999,South Asia,Rajastán,CLOSED,Smart watch,1/17/2018 12:06,Standard Class
3,22.860001,304.809998,Advance shipping,0,Sporting Goods,Los Angeles,EE. UU.,Tana,19490,Tate,...,1,327.75,304.809998,22.860001,Oceania,Queensland,COMPLETE,Smart watch,1/16/2018 11:45,Standard Class
4,134.210007,298.250000,Advance shipping,0,Sporting Goods,Caguas,Puerto Rico,Orli,19489,Hendricks,...,1,327.75,298.250000,134.210007,Oceania,Queensland,PENDING_PAYMENT,Smart watch,1/15/2018 11:24,Standard Class


CONDITIONAL COLUMNS

In [8]:
# Classify the profit 
conditions = [
     df["Order Profit Per Order"] > 100 ,
     df["Order Profit Per Order"] >= 0 ,
     df["Order Profit Per Order"] < 0 
]

choices = [
     "High Profit" ,
     "Normal Profit",
     "Loss"
]

df["Category Profit"] = np.select(
     conditions,
     choices,
     default="Unknown"
)

df["Category Profit"].value_counts()

Category Profit
Normal Profit    126425
Loss              33784
High Profit       20310
Name: count, dtype: int64

DISCOUNT CLASSIFICATION

In [10]:
df["Order Profit Per Order"].describe()

count    180519.000000
mean         21.974989
std         104.433526
min       -4274.979980
25%           7.000000
50%          31.520000
75%          64.800003
max         911.799988
Name: Order Profit Per Order, dtype: float64

In [11]:
conditions = [
     df["Order Profit Per Order"] < 0.10 ,
     df["Order Profit Per Order"] <= 0.20 ,
     df["Order Profit Per Order"] > 0.20 
]

choices = [
     "Low Discount" ,
     "Medium Discount" , 
     "High Discount"
]

df["Discount Category"] = np.select(
     conditions ,
     choices ,
     default="Unknown"
)

df["Discount Category"].value_counts()

Discount Category
High Discount      145544
Low Discount        34962
Medium Discount        13
Name: count, dtype: int64

RANKING

In [28]:
df["Customer Name"] = (
     df["Customer Fname"] + " " + df["Customer Lname"]
)

In [30]:
top_10_customers = (
     df.groupby("Customer Name")["Sales"]
     .sum().sort_values(ascending=False).head(10)
)

top_10_customers

Customer Name
Mary Smith           4.771646e+06
Robert Smith         1.522310e+05
James Smith          1.479244e+05
David Smith          1.457768e+05
John Smith           1.298826e+05
William Smith        1.085233e+05
Michael Smith        9.284303e+04
Elizabeth Smith      9.051602e+04
Mary Jones           8.886048e+04
Christopher Smith    8.527105e+04
Name: Sales, dtype: float64

In [36]:
# ranking top products
top_10_products = (
     df.groupby("Product Name")[["Sales" , "Order Profit Per Order"]]
     .sum().sort_values(by="Sales",ascending=False).head(10)
)
top_10_products

,Sales,Order Profit Per Order
Product Name,,
Field & Stream Sportsman 16 Gun Fire Safe,6.929654e+06,756220.767190
Perfect Fitness Perfect Rip Deck,4.421143e+06,493828.299782
Diamondback Women's Serene Classic Comfort Bi,4.118426e+06,427455.568106
Nike Men's Free 5.0+ Running Shoe,3.667633e+06,379915.818503
Nike Men's Dri-FIT Victory Golf Polo,3.147800e+06,350421.029567
Pelican Sunstream 100 Kayak,3.099845e+06,324076.370020
Nike Men's CJ Elite 2 TD Football Cleat,2.891758e+06,311902.820214
O'Brien Men's Neoprene Life Vest,2.888994e+06,318451.430554
Under Armour Girls' Toddler Spine Surge Runni,1.269083e+06,126278.510299


TOP SELLING VS PROFITABLE PRODUCTS

In [38]:
# most profitable product among top 10
top_10_products["Order Profit Per Order"].idxmax()

'Field & Stream Sportsman 16 Gun Fire Safe'

In [39]:
# profit value of profitable product
top_10_products["Order Profit Per Order"].max()

756220.767189502

In [40]:
# least profitable product
top_10_products["Order Profit Per Order"].idxmin()

'Dell Laptop'

In [41]:
# profit value od least profitable product
top_10_products["Order Profit Per Order"].min()

69656.81017112

HIGH SALES BUT LOW PROFIT

In [43]:
product_analysis = (
     df.groupby("Product Name")[["Sales" , "Order Profit Per Order"]]
     .sum()
)
product_analysis

,Sales,Order Profit Per Order
Product Name,,
Adult dog supplies,41524.800753,3589.259959
Baby sweater,12229.560379,1525.029992
Bag Boy Beverage Holder,21116.549776,3173.780008
Bag Boy M330 Push Cart,16637.919929,2969.110042
Bowflex SelectTech 1090 Dumbbells,5999.899902,1190.779995
...,...,...
adidas Kids' F5 Messi FG Soccer Cleat,27327.190540,2906.330037
adidas Men's F10 Messi TRX FG Soccer Cleat,56330.611645,8213.070031
adidas Men's Germany Black Crest Away Tee,21475.000000,2833.720001


In [51]:
high_sales_loss_products = product_analysis[
     (product_analysis["Sales"] > 100000) &
     (product_analysis["Order Profit Per Order"] < 0) 
]

high_sales_loss_products.sort_values(
     by="Sales" , ascending=False
)

,Sales,Order Profit Per Order
Product Name,,


In [60]:
# find top 5 products by profit
top_5_profit_products = (
     product_analysis.sort_values(
          by="Order Profit Per Order",ascending=False).head(5)
)

top_5_profit_products

,Sales,Order Profit Per Order
Product Name,,
Field & Stream Sportsman 16 Gun Fire Safe,6.929654e+06,756220.767190
Perfect Fitness Perfect Rip Deck,4.421143e+06,493828.299782
Diamondback Women's Serene Classic Comfort Bi,4.118426e+06,427455.568106
Nike Men's Free 5.0+ Running Shoe,3.667633e+06,379915.818503
Nike Men's Dri-FIT Victory Golf Polo,3.147800e+06,350421.029567


In [58]:
# find top 5 products by sales
top_5_sales_products = (
     product_analysis.sort_values(
          by="Sales",ascending=False).head(5)
     )
top_5_sales_products

,Sales,Order Profit Per Order
Product Name,,
Field & Stream Sportsman 16 Gun Fire Safe,6.929654e+06,756220.767190
Perfect Fitness Perfect Rip Deck,4.421143e+06,493828.299782
Diamondback Women's Serene Classic Comfort Bi,4.118426e+06,427455.568106
Nike Men's Free 5.0+ Running Shoe,3.667633e+06,379915.818503
Nike Men's Dri-FIT Victory Golf Polo,3.147800e+06,350421.029567
